# Entrainer un modèle de classification

Nous allons entraîner un modèle de classification d'images capable de trier les articles de presse en fonction de la présence d'une illustration. Nous utiliserons notamment un modèle de classification YOLO de taille minimale.

In [3]:
%pip install scikit-learn
%pip install ultralytics
%pip install tqdm
%pip install pandas


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import requests
import random
import os
import time
import shutil

from sklearn.model_selection import train_test_split # Permet de spliter facilement un dataset 
from tqdm import tqdm

## 1. Récupération des annotations

Un classificateur catégorise les documents d'un ensemble de données. Parmi les classificateurs, on trouve le classificateur multi-étiquettes supervisé.

Nous avons répartit les documents en classes prédéfinies :

- Avec illustration
- Sans illustration

Ici, les annotations correspondent aux dossiers où sont stockées les images. Les classificateurs sont des modèles qui attribuent une étiquette à l'image entière, ce qui simplifie grandement la génération de données.

In [5]:
!apt-get install -y git-lfs > /dev/null
!git lfs install

!rm -rf formationVision
!git clone https://huggingface.co/datasets/Anarchiviste/formationVision

%cd formationVision
!git lfs pull
%cd ..

E: Impossible d'ouvrir le fichier verrou /var/lib/dpkg/lock-frontend - open (13: Permission non accordée)
E: Impossible d'obtenir le verrou de dpkg (/var/lib/dpkg/lock-frontend). Avez-vous les droits du superutilisateur ?
Updated git hooks.
Git LFS initialized.
Clonage dans 'formationVision'...
remote: Enumerating objects: 268, done.
remote: Total 268 (delta 0), reused 0 (delta 0), pack-reused 268 (from 1)
Réception d'objets: 100% (268/268), 41.34 Kio | 4.13 Mio/s, fait.
Résolution des deltas: 100% (6/6), fait.
Filtrage du contenu: 100% (238/238), 67.32 Mio | 9.92 Mio/s, fait.
/home/jmusquin/Cnrs-formation-vision/Corrigés/formationVision
/home/jmusquin/Cnrs-formation-vision/Corrigés


In [6]:
source = "./formationVision"

illustrations = os.listdir(os.path.join(source, 'illustrated/'))
textOnly = os.listdir(os.path.join(source, 'textonly/'))

trainIllustrations, valIllustrations = train_test_split(illustrations, train_size=0.8, random_state=42) # Nous déterminons ici la taille du jeu d'entraînement (80% des images totales) et la seed pour un aléatoire reproductible
trainTextOnly, valTextOnly = train_test_split(textOnly, train_size=0.8, random_state=42)


In [7]:
!rm -rf dataset/
!mkdir -p ./dataset/train/illustrated/
!mkdir -p ./dataset/train/textonly/
!mkdir -p ./dataset/val/illustrated/
!mkdir -p ./dataset/val/textonly/


In [8]:
print(f"Nous aurons {len(trainIllustrations)} illustrations dans Train et {len(valIllustrations)} illustrations dans Val")
print(f"Nous aurons {len(trainTextOnly)} textes dans Train et {len(valTextOnly)} textes dans Val")

Nous aurons 72 illustrations dans Train et 18 illustrations dans Val
Nous aurons 76 textes dans Train et 19 textes dans Val


In [9]:
copies = [
    (trainIllustrations, 'illustrated', './dataset/train/illustrated/'),
    (valIllustrations, 'illustrated', './dataset/val/illustrated/'),
    (trainTextOnly, 'textonly', './dataset/train/textonly/'),
    (valTextOnly, 'textonly', './dataset/val/textonly/')
]

# Mise en place d'un dataset de Train et de Val en attribuant les images à leurs dossiers
for files, folder_name, dest_dir in copies:
    for file_name in files:
        shutil.copy(
            os.path.join(source, folder_name, file_name),
            os.path.join(dest_dir, file_name)
        )

In [10]:
from ultralytics import YOLO

# Nous importons un modèle Yolo 26 nano classificateur avec la bibliothèque d'Ultralytics
model = YOLO("yolo26n-cls.pt")

In [ ]:
dataset = "./dataset/"
model.train(
    data=dataset,
    flipud=0, fliplr=0,
    epochs=10, imgsz=640, # attributions des hyperparamètres : 10 cycles, les images sont en tailles 640x640
    project="runs/classify",   # chemin relatif au dossier courant
    name="train",
)

New https://pypi.org/project/ultralytics/8.4.212 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.201 🚀 Python-3.13.18 torch-2.15.0+cu132 CPU (Intel Core i5-8250U 1.60GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./dataset/, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0, flipud=0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-cls.pt, momentum=0.937, mosai

# Faire tourner une inférence

L'inférence en intelligence artificielle est le processus par lequel un modèle entraîné applique les connaissances acquises pour analyser de nouvelles données et produire une prédiction, une classification ou une réponse.

In [15]:
# Yolo rend 2 modèles (.pt), un nommé best.pt et un last.pt. Sur de longs cycles d'entraînement il faut éviter le surapprentissage, on charge donc le modèle avec les meilleurs valeures.
best = YOLO("./runs/classify/train/weights/best.pt")

FileNotFoundError: [Errno 2] No such file or directory: 'runs/classify/train/weights/best.pt'

In [20]:
import os
import pandas as pd

# Dossier contenant les images de test
test_dir = "./formationVision/test/press/"

results_list = []
image_files = [f for f in os.listdir(test_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

# Exécuter l'inférence sur chaque image
for file_name in image_files:
    file_path = os.path.join(test_dir, file_name)
    results = best(file_path, verbose=False)

    for r in results:
        probs = r.probs
        top_class_idx = probs.top1
        top_class_name = r.names[top_class_idx]
        top_class_conf = float(probs.top1conf)

        results_list.append({
            "filename": file_name,
            "prediction": top_class_name,
            "confidence": top_class_conf
        })

# Convertir en DataFrame pandas et sauvegarder en CSV
df = pd.DataFrame(results_list)
df.sort_values(by='filename', ascending=False)
df.to_csv("predictions.csv", index=False)

print(f"Inférence terminée ! Les résultats ont été enregistrés dans 'predictions.csv'.")
print(df.head())

Inférence terminée ! Les résultats ont été enregistrés dans 'predictions.csv'.
  filename   prediction  confidence
0    4.png  illustrated    0.910941
1   37.png     textonly    0.986176
2   16.png     textonly    0.999589
3   14.png     textonly    0.981780
4   34.png     textonly    0.979701
